# Notebook 04: Decoding Algorithms

## Why This Matters
Token-level sampling (notebook 03) decides how to pick *one* token at a time. Decoding algorithms decide the *strategy* for building the full sequence. Beam search tries to find the highest-probability complete sequence. Greedy finds a locally optimal one. These algorithms make fundamentally different tradeoffs between quality, diversity, and compute.

Understanding decoding algorithms matters for:
- Machine translation and summarization (beam search is still standard)
- Why LLM chat APIs use sampling rather than beam search (hallucination, diversity)
- Speculative decoding (notebook 10) — which builds on the greedy/sampling distinction
- Constrained generation (notebook 15) — which extends beam search

## Structure
1. Greedy decoding — recap with sequence-level view
2. Beam search — the standard for seq2seq
3. Why beam search fails for open-ended generation
4. Stochastic sampling as decoding
5. Contrastive search — a principled alternative
6. Best-of-N sampling
7. Compute cost comparison

## What You'll Learn
- Why beam search can produce degenerate (repetitive) text at high beam widths
- The exposure bias problem and why it matters
- Why best-of-N is a surprisingly strong baseline for reasoning tasks
- The compute tradeoff: beam search vs. sampling vs. best-of-N

## Background

### What is a decoding algorithm?

Token-level sampling (notebook 03) decides *how* to pick the next token from a probability distribution. A decoding *algorithm* decides the *strategy* for building the full output sequence — it operates at a higher level, potentially maintaining multiple candidate sequences, looking ahead, or re-ranking outputs after generation.

The distinction matters because the token that looks most likely at step `t` might not lead to the best overall sequence by step `t+100`. Decoding algorithms trade off computation against the quality of the final result.

### Why beam search became the standard — and then fell out of favor

**Beam search** was the dominant decoding algorithm for neural sequence models from ~2015 to ~2020. It maintains `k` candidate sequences ("beams") at each step, expanding each to all possible next tokens and keeping the `k` highest-scoring sequences. This is a tractable approximation to finding the globally most-probable sequence.

Beam search worked extremely well for **constrained** generation tasks: machine translation (one correct target sentence), abstractive summarization (faithful compression of a source), image captioning. For these tasks, maximizing sequence probability is a reasonable proxy for quality.

But when researchers applied beam search to open-ended text generation, something surprising happened: **higher beam widths made output worse**, not better. Wieting et al. and Meister et al. (2020-2022) showed that the highest-probability sequences tend to be degenerate — repetitive, generic, hedged. Beam search finds and exploits whatever repetitive patterns the model assigns high probability to. This is why large language models in chat mode almost universally use sampling rather than beam search.

### The modern landscape

Today's production LLM systems converged on a clear division:

- **Sampling (nucleus + temperature)**: the default for open-ended generation — chat, creative writing, code generation. Stochastic but produces diverse, human-like text.
- **Beam search**: still used for machine translation and strict summarization tasks where a single "best" output is required and diversity is undesirable.
- **Best-of-N with verifier**: the emerging frontier for reasoning and code. Generate N independent samples, then pick the one that satisfies some external criterion (passes tests, gives the most common answer, scores highest on a reward model). Shown to be surprisingly competitive with much more complex approaches.

### Speculative decoding preview

One important algorithm is conspicuously absent from this notebook: **speculative decoding** (notebook 10). It's architecturally different from the algorithms here — it uses a separate small "draft" model to propose tokens that a large "target" model then accepts or rejects. It deserves its own notebook because it's the most practically impactful decoding innovation for production inference since the KV cache.

In [ ]:
%pip install -q torch transformers

In [ ]:
import math
import heapq
import torch
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

torch.manual_seed(42)
print("Imports OK")

## 1. The Decoding Problem

Given a model that produces `P(token | context)`, find the sequence `y₁, y₂, ..., yₙ` that maximizes:

$$P(y_1, y_2, ..., y_n) = \prod_{t=1}^{n} P(y_t | y_1, ..., y_{t-1}, x)$$

Exact search is intractable — vocabulary size `V` raised to sequence length `n` candidates. Decoding algorithms are tractable approximations.

```
Vocabulary size:  50,000
Sequence length:  50 tokens
Exact candidates: 50,000^50 ≈ 10^230  (impossibly large)

Greedy:     1 path, 1× model calls per step
Beam (k=4): 4 paths, 4× model calls per step
Sampling:   1 path, 1× model calls per step (but stochastic)
Best-of-N:  N paths, N× total model calls
```

In [ ]:
# Toy language model for illustrating decoding algorithms
# Uses a simple bigram model over a small vocabulary

VOCAB = ["<bos>", "the", "cat", "sat", "on", "mat", "dog", "ran", "fast", "slowly", "<eos>"]
V = len(VOCAB)
tok2id = {w: i for i, w in enumerate(VOCAB)}
id2tok = {i: w for i, w in enumerate(VOCAB)}

# Hand-crafted transition logits: [from_token, to_token]
# Higher = more likely. This encodes: "the cat sat on the mat" is most likely
import torch
transitions = torch.full((V, V), -5.0)  # base: everything very unlikely

# <bos> → the (0.9), dog (0.1)
transitions[tok2id["<bos>"], tok2id["the"]] = 5.0
transitions[tok2id["<bos>"], tok2id["dog"]] = 2.0

# the → cat (0.7), mat (0.3)
transitions[tok2id["the"], tok2id["cat"]] = 4.0
transitions[tok2id["the"], tok2id["mat"]] = 2.0

# cat → sat (0.8), ran (0.2)
transitions[tok2id["cat"], tok2id["sat"]] = 4.5
transitions[tok2id["cat"], tok2id["ran"]] = 2.0

# sat → on (0.9), slowly (0.1)
transitions[tok2id["sat"], tok2id["on"]] = 5.0
transitions[tok2id["sat"], tok2id["slowly"]] = 1.5

# on → the (1.0)
transitions[tok2id["on"], tok2id["the"]] = 5.0

# mat → <eos> (0.9), . (0.1)
transitions[tok2id["mat"], tok2id["<eos>"]] = 5.0

# dog → ran (0.8), sat (0.2)
transitions[tok2id["dog"], tok2id["ran"]] = 4.0
transitions[tok2id["dog"], tok2id["sat"]] = 1.5

# ran → fast (0.7), slowly (0.3)
transitions[tok2id["ran"], tok2id["fast"]] = 3.5
transitions[tok2id["ran"], tok2id["slowly"]] = 2.5

# fast/slowly → <eos>
transitions[tok2id["fast"], tok2id["<eos>"]] = 5.0
transitions[tok2id["slowly"], tok2id["<eos>"]] = 5.0

def get_logits(token_id: int) -> torch.Tensor:
    """Return logits for next token given current token."""
    return transitions[token_id]

def decode_ids(ids: List[int]) -> str:
    return " ".join(id2tok[i] for i in ids if id2tok[i] not in ["<bos>", "<eos>"])

print("Toy vocabulary:", VOCAB)
print("\nExpected most-likely sequence: 'the cat sat on the mat'")

## 2. Greedy Decoding

At each step, pick the single most likely token. Fast, deterministic, locally optimal but globally suboptimal.

```
Step 1: P(the|<bos>)=0.88, P(dog|<bos>)=0.12  → pick "the"
Step 2: P(cat|the)=0.73,   P(mat|the)=0.27    → pick "cat"
Step 3: P(sat|cat)=0.85,   P(ran|cat)=0.15    → pick "sat"
...
```

The path `the → cat → sat → on → the → mat` is found because each local maximum happens to be the global maximum in this toy example. This doesn't hold in general.

In [ ]:
def greedy_decode(max_len: int = 10) -> Tuple[List[int], float]:
    """Greedy decoding. Returns (token_ids, log_prob)."""
    current = tok2id["<bos>"]
    sequence = [current]
    log_prob = 0.0

    for step in range(max_len):
        logits = get_logits(current)
        probs = F.softmax(logits, dim=-1)
        next_tok = torch.argmax(probs).item()
        step_log_prob = torch.log(probs[next_tok]).item()

        print(f"  Step {step+1}: top candidates: "
              f"{[(id2tok[i], f'{probs[i].item():.3f}') for i in probs.topk(3).indices.tolist()]}")
        print(f"           → chose '{id2tok[next_tok]}' (log_p={step_log_prob:.3f})")

        sequence.append(next_tok)
        log_prob += step_log_prob

        if next_tok == tok2id["<eos>"]:
            break
        current = next_tok

    return sequence, log_prob


print("Greedy decoding:")
seq, lp = greedy_decode()
print(f"\nResult: '{decode_ids(seq)}'")
print(f"Log probability: {lp:.3f}  (prob = {math.exp(lp):.6f})")

## 3. Beam Search

Beam search maintains `k` candidate sequences ("beams") at each step. For each beam, it expands to all `V` possible next tokens, then keeps the `k` highest-scoring sequences.

```
Beam width k=2, step-by-step:

Step 0:  Beams: [(<bos>,  score=0)]
Step 1:  Expand: [(<bos>,the, -0.13), (<bos>,dog, -2.1), ...]
         Keep top-2: [the(-0.13), dog(-2.1)]
Step 2:  Expand the: [(the,cat,-0.50), (the,mat,-1.44), ...]
         Expand dog: [(dog,ran,-0.60), (dog,sat,-1.9), ...]
         Keep top-2 globally: [(the,cat,-0.50), (dog,ran,-0.60)]
...
```

Score = sum of log-probabilities (log-space multiplication). Beams that reach `<eos>` are finalized.

In [ ]:
@dataclass
class Beam:
    tokens: List[int]
    log_prob: float
    is_done: bool = False

    def score(self, length_penalty: float = 1.0) -> float:
        """Length-normalized score. Prevents beam search from preferring short sequences."""
        length = max(len(self.tokens), 1)
        return self.log_prob / (length ** length_penalty)


def beam_search(
    beam_width: int = 3,
    max_len: int = 10,
    length_penalty: float = 1.0,
    verbose: bool = True,
) -> List[Beam]:
    """Beam search decoding. Returns completed beams sorted by score."""
    bos = tok2id["<bos>"]
    eos = tok2id["<eos>"]

    # Initialize beams
    active_beams = [Beam(tokens=[bos], log_prob=0.0)]
    completed_beams = []

    for step in range(max_len):
        if not active_beams:
            break

        # Expand each active beam
        candidates = []
        for beam in active_beams:
            last_tok = beam.tokens[-1]
            logits = get_logits(last_tok)
            log_probs = F.log_softmax(logits, dim=-1)

            # Add top-V candidates from this beam
            for tok_id in range(V):
                new_log_prob = beam.log_prob + log_probs[tok_id].item()
                new_beam = Beam(
                    tokens=beam.tokens + [tok_id],
                    log_prob=new_log_prob,
                    is_done=(tok_id == eos)
                )
                candidates.append(new_beam)

        # Sort by score and keep top beam_width
        candidates.sort(key=lambda b: b.score(length_penalty), reverse=True)
        top_candidates = candidates[:beam_width * 2]  # extra buffer

        # Separate finished and active
        active_beams = []
        for beam in top_candidates:
            if beam.is_done:
                completed_beams.append(beam)
            elif len(active_beams) < beam_width:
                active_beams.append(beam)

        if verbose:
            print(f"  Step {step+1} active beams:")
            for i, b in enumerate(active_beams[:3]):
                print(f"    [{i+1}] '{decode_ids(b.tokens)}'  score={b.score(length_penalty):.3f}")

    # Add any remaining active beams to completed
    completed_beams.extend(active_beams)
    completed_beams.sort(key=lambda b: b.score(length_penalty), reverse=True)
    return completed_beams


print("Beam search (width=3):")
beams = beam_search(beam_width=3, verbose=True)
print(f"\nTop sequences:")
for i, b in enumerate(beams[:4]):
    print(f"  [{i+1}] '{decode_ids(b.tokens)}'  score={b.score():.3f}  log_prob={b.log_prob:.3f}")

## 4. Why Beam Search Fails for Open-Ended Generation

Beam search optimizes sequence probability — exactly what you want for translation (one correct answer). But for open-ended text generation, this produces:

1. **Degenerate repetition**: The highest-probability continuation of many contexts is to repeat what was just said. Beam search finds and exploits this.

2. **Bland text**: Maximum-probability text tends to be generic and safe. The model assigns high probability to common, expected phrases.

3. **Length bias**: Without length normalization, beam search strongly prefers shorter sequences (lower cumulative log-prob penalty from fewer multiplications).

Holtzman et al. (2020) showed that human text is **not** the most probable text — it lives in a moderate-probability region. Their fix: nucleus sampling (notebook 03).

In [ ]:
# Demonstrate length penalty effect
print("Effect of length penalty on beam search results:")
for lp in [0.0, 0.6, 1.0, 1.5]:
    beams = beam_search(beam_width=3, length_penalty=lp, verbose=False)
    top = beams[0]
    print(f"  length_penalty={lp:.1f}: '{decode_ids(top.tokens)}'  "
          f"(len={len(top.tokens)}, score={top.score(lp):.3f})")

print()
print("lp=0.0: no penalty, beam search ignores length")
print("lp=1.0: divide by length (standard normalization)")
print("lp>1.0: length bonus — encourages longer sequences")

## 5. Diverse Beam Search

Standard beam search tends to return near-identical sequences (all slight variants of the most probable). **Diverse beam search** (Vijayakumar et al., 2016) divides beams into groups and penalizes similarity between groups.

```
Groups: G=2, beams per group: k/G=2

Group 1: searches normally
Group 2: at each step, penalizes tokens chosen by Group 1
            → forced to find a different path
```

Used in: multiple hypothesis generation, question answering (return N diverse answers), creative writing where you want variety.

In [ ]:
def diverse_beam_search(
    num_groups: int = 2,
    beams_per_group: int = 2,
    diversity_penalty: float = 1.5,
    max_len: int = 10,
) -> List[List[Beam]]:
    """Simplified diverse beam search."""
    bos = tok2id["<bos>"]
    eos = tok2id["<eos>"]

    # Initialize one set of beams per group
    groups = [[Beam(tokens=[bos], log_prob=0.0)] for _ in range(num_groups)]
    completed = [[] for _ in range(num_groups)]

    for step in range(max_len):
        tokens_chosen_by_prev_groups = []  # track tokens other groups picked

        for g_idx, group_beams in enumerate(groups):
            if not group_beams:
                continue

            candidates = []
            for beam in group_beams:
                last_tok = beam.tokens[-1]
                logits = get_logits(last_tok).clone()

                # Apply diversity penalty for tokens chosen by earlier groups
                for prev_tok in tokens_chosen_by_prev_groups:
                    logits[prev_tok] -= diversity_penalty

                log_probs = F.log_softmax(logits, dim=-1)
                for tok_id in range(V):
                    new_lp = beam.log_prob + log_probs[tok_id].item()
                    candidates.append(Beam(
                        tokens=beam.tokens + [tok_id],
                        log_prob=new_lp,
                        is_done=(tok_id == eos)
                    ))

            candidates.sort(key=lambda b: b.log_prob, reverse=True)

            new_group = []
            for beam in candidates:
                if beam.is_done:
                    completed[g_idx].append(beam)
                elif len(new_group) < beams_per_group:
                    new_group.append(beam)
                    # Record the token this group chose (for diversity penalty)
                    tokens_chosen_by_prev_groups.append(beam.tokens[-1])

            groups[g_idx] = new_group

    # Collect remaining
    for g_idx, group_beams in enumerate(groups):
        completed[g_idx].extend(group_beams)

    return completed


print("Diverse beam search (2 groups, penalty=1.5):")
groups = diverse_beam_search(num_groups=2, beams_per_group=2, diversity_penalty=1.5)
for g_idx, group in enumerate(groups):
    print(f"\nGroup {g_idx+1}:")
    for b in sorted(group, key=lambda x: x.log_prob, reverse=True)[:3]:
        print(f"  '{decode_ids(b.tokens)}'  log_prob={b.log_prob:.3f}")

## 6. Contrastive Search

Contrastive search (Su et al., 2022) is a decoding method designed to avoid degenerate repetition without sacrificing coherence. It balances:

1. **Confidence**: the model's predicted probability for the next token
2. **Degeneration penalty**: how similar the candidate token's representation is to tokens already in context

$$\text{score}(y_t) = (1-\alpha) \cdot p(y_t | x, y_{<t}) - \alpha \cdot \max_{j<t} \cos(h_{y_t}, h_{y_j})$$

Where `h` is the hidden state (representation) of the token. High cosine similarity to previous tokens means the candidate is "too similar" to what came before — penalize it.

`α` controls the balance: 0 = standard greedy, 1 = pure anti-repetition.

We'll demonstrate the concept with simulated hidden states.

In [ ]:
import torch.nn.functional as F

# Simulate hidden states for each vocabulary token (in practice, from the model)
torch.manual_seed(7)
hidden_dim = 16
token_hiddens = torch.randn(V, hidden_dim)  # (vocab, hidden_dim)
token_hiddens = F.normalize(token_hiddens, dim=-1)  # unit vectors

def contrastive_score(
    probs: torch.Tensor,
    candidate_ids: List[int],
    context_ids: List[int],
    alpha: float = 0.6,
) -> int:
    """
    Score candidates by model confidence minus degeneration penalty.
    Returns the best candidate token id.
    """
    if not context_ids:
        # No context yet, just pick highest probability
        return max(candidate_ids, key=lambda t: probs[t].item())

    context_hiddens = token_hiddens[context_ids]  # (context_len, hidden)

    best_tok, best_score = None, float('-inf')
    for tok in candidate_ids:
        confidence = (1 - alpha) * probs[tok].item()

        # Degeneration penalty: max cosine similarity to any context token
        h_tok = token_hiddens[tok].unsqueeze(0)        # (1, hidden)
        sims = F.cosine_similarity(h_tok, context_hiddens, dim=-1)  # (context_len,)
        degeneration = alpha * sims.max().item()

        score = confidence - degeneration
        if score > best_score:
            best_score = score
            best_tok = tok
    return best_tok


def contrastive_decode(k: int = 4, alpha: float = 0.6, max_len: int = 10) -> List[int]:
    """Contrastive search: choose top-k candidates, score by confidence - degeneration."""
    current = tok2id["<bos>"]
    sequence = [current]

    for step in range(max_len):
        logits = get_logits(current)
        probs = F.softmax(logits, dim=-1)

        # Get top-k candidates
        top_k_ids = probs.topk(k).indices.tolist()

        # Score by confidence - degeneration penalty
        next_tok = contrastive_score(probs, top_k_ids, sequence[1:], alpha)

        sequence.append(next_tok)
        if next_tok == tok2id["<eos>"]:
            break
        current = next_tok

    return sequence


print("Contrastive search with different alpha values:")
for alpha in [0.0, 0.3, 0.6, 0.9]:
    seq = contrastive_decode(k=4, alpha=alpha)
    print(f"  alpha={alpha:.1f}: '{decode_ids(seq)}'")

print()
print("alpha=0.0: pure confidence (greedy)")
print("alpha=0.6: balanced (Su et al. default)")
print("alpha=0.9: heavy anti-repetition")

## 7. Best-of-N Sampling

Generate `N` independent sequences using sampling, then re-rank them by some scoring function. The scoring function can be:

- **Model log-probability**: pick the highest-probability sample (approximates beam search with more diversity)
- **Reward model score**: used in RLHF alignment
- **Self-consistency**: for reasoning tasks, pick the answer that appears most frequently across N samples
- **Verifier**: for math/code, run a verifier and pick the first passing solution

Best-of-N is a surprisingly strong baseline for reasoning tasks (Brown et al., 2024 showed it's competitive with more complex approaches).

In [ ]:
def sample_sequence(max_len: int = 10, temperature: float = 0.8) -> Tuple[List[int], float]:
    """Sample one sequence using temperature sampling."""
    current = tok2id["<bos>"]
    sequence = [current]
    log_prob = 0.0

    for _ in range(max_len):
        logits = get_logits(current) / temperature
        probs = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1).item()
        log_prob += torch.log(probs[next_tok]).item()
        sequence.append(next_tok)
        if next_tok == tok2id["<eos>"]:
            break
        current = next_tok

    return sequence, log_prob


def best_of_n(
    n: int,
    temperature: float = 0.8,
    scoring: str = "log_prob",  # "log_prob" or "self_consistency"
) -> Tuple[List[int], List[Tuple[str, float]]]:
    """Sample N sequences and re-rank."""
    samples = [sample_sequence(temperature=temperature) for _ in range(n)]

    if scoring == "log_prob":
        # Pick highest log-probability sample
        best = max(samples, key=lambda x: x[1])
        all_results = [(decode_ids(s), f"{lp:.3f}") for s, lp in
                       sorted(samples, key=lambda x: x[1], reverse=True)]
    elif scoring == "self_consistency":
        # Pick most frequent sequence (for closed-form answers)
        texts = [decode_ids(s) for s, _ in samples]
        from collections import Counter
        counts = Counter(texts)
        best_text = counts.most_common(1)[0][0]
        best = next((s, lp) for s, lp in samples if decode_ids(s) == best_text)
        all_results = [(t, str(c)) for t, c in counts.most_common()]

    return best[0], all_results


print(f"Best-of-N sampling (N=10, temperature=0.8):")
best_seq, all_seqs = best_of_n(n=10, temperature=0.8, scoring="log_prob")
print(f"\nAll samples (sorted by log-prob):")
for text, score in all_seqs:
    print(f"  '{text}'  log_prob={score}")
print(f"\nBest-of-10 result: '{decode_ids(best_seq)}'")

print(f"\nSelf-consistency (N=20):")
_, counts = best_of_n(n=20, temperature=1.0, scoring="self_consistency")
print("Vote counts:")
for text, count in counts:
    print(f"  '{text}': {count} votes")

## 8. Compute Cost Comparison

Each decoding strategy has a different compute profile:

```
Strategy          | Prefill cost | Decode cost per step | Total at length L
------------------|--------------|----------------------|------------------
Greedy            | 1× forward   | 1× forward           | L × forward
Beam (width=k)    | 1× forward   | k× forward           | L × k × forward
Sampling          | 1× forward   | 1× forward           | L × forward
Best-of-N         | N× forward   | N× forward           | L × N × forward
Contrastive       | 1× forward   | 1× forward + cosine  | ≈ L × forward
```

Best-of-N is embarrassingly parallel — all N sequences can be batched together in one model call. In practice, `best_of_n=N` is implemented as a batch of N sequences with batch size N.

In [ ]:
# Theoretical FLOP comparison (relative units)
configs = [
    ("Greedy",           1, 1,  "deterministic"),
    ("Beam search k=4",  4, 1,  "highest prob sequence"),
    ("Beam search k=8",  8, 1,  "higher quality, more cost"),
    ("Sampling",         1, 1,  "diverse, stochastic"),
    ("Best-of-4",        4, 4,  "parallelizable"),
    ("Best-of-16",      16, 16, "strong for reasoning"),
]

seq_len = 100
print(f"Relative compute cost (sequence length={seq_len}):")
print(f"{'Strategy':<22} {'Parallel?':>10} {'Relative cost':>15} {'Use case'}")
print("-" * 75)

base = seq_len  # greedy baseline
for name, width, parallel_factor, use_case in configs:
    cost = seq_len * width
    parallel_batches = cost / parallel_factor  # if batched
    print(f"{name:<22} {'Yes' if parallel_factor > 1 else 'No':>10} "
          f"{cost/base:>12.0f}×  {use_case}")

## 9. Practical Guidance: When to Use Each Algorithm

| Task | Best algorithm | Why |
|------|---------------|-----|
| Machine translation | Beam search (k=4-6) | Single correct target, probability maximization appropriate |
| Summarization | Beam search or sampling | Depends: faithful summary → beam; creative → sample |
| Chat / assistant | Temperature sampling + top-p | Diversity needed; beam search produces bland text |
| Code generation | Sampling + best-of-N + verifier | Sample diverse solutions, pick one that passes tests |
| Math reasoning | Best-of-N + self-consistency | Sample N chains, majority-vote the final answer |
| Structured output | Greedy + constrained decoding | Exact format required; see notebook 15 |
| Creative writing | Sampling (high temp + nucleus) | Diversity is the goal |

**The modern consensus:** For LLM chat/generation, sampling + nucleus is the standard. Beam search is largely confined to seq2seq tasks (translation, summarization with strict fidelity requirements). Best-of-N with a verifier or reward model is the frontier for reasoning and code.

In [ ]:
# Summary: compare all methods on the toy model
print("All decoding methods on the toy model:")
print("-" * 55)

# Greedy
seq, lp = greedy_decode(max_len=10)
print(f"Greedy:             '{decode_ids(seq)}'  (lp={lp:.2f})")

# Beam search
beams = beam_search(beam_width=3, verbose=False)
print(f"Beam (k=3, best):   '{decode_ids(beams[0].tokens)}'  (score={beams[0].score():.2f})")

# Sampling
torch.manual_seed(0)
seq_s, lp_s = sample_sequence(temperature=0.8)
print(f"Sampling (T=0.8):   '{decode_ids(seq_s)}'  (lp={lp_s:.2f})")

# Best-of-N
torch.manual_seed(1)
best_seq, _ = best_of_n(n=5, temperature=0.8)
print(f"Best-of-5:          '{decode_ids(best_seq)}'")

# Contrastive
cont_seq = contrastive_decode(k=4, alpha=0.6)
print(f"Contrastive (a=0.6):'{decode_ids(cont_seq)}'")

## Summary

| Algorithm | Finds | Cost | Diversity | Best for |
|-----------|-------|------|-----------|----------|
| Greedy | Local optimum | 1× | None | Fast, deterministic |
| Beam search | Near-global optimum | k× | Low | Translation, structured |
| Sampling | Random trajectory | 1× | High | Chat, creative generation |
| Contrastive | High-prob + diverse | ≈1× | Medium | Long-form, anti-repetition |
| Best-of-N | Best of N samples | N× | High | Reasoning, code with verifier |

**Next:** Notebook 05 covers the KV cache — the memory structure that makes autoregressive inference tractable, and the source of most memory pressure in LLM serving.